# Buổi 6 — Content Engine: Demo Live E2E qua n8n Webhook thật

Notebook này gọi **webhook n8n thật** (không mock), giống tinh thần `04_contract_review_lab_demo.ipynb` /
`05_cskh_bot_lab_demo.ipynb` — nhưng khác 2 điểm có lý do:

1. **Không tự-launch n8n cục bộ được.** Workflow B5 định tuyến bằng Code node JS thuần (tất định, miễn phí) nên
   tự-chứa được trong docker-compose dùng-rồi-bỏ. Workflow B6 gọi **LLM thật (Gemini) + sinh ảnh thật
   (GeminiGen.ai trả phí)** — không tự động hoá được credential OAuth/API key trong container tạm. Notebook này
   gọi thẳng 3 webhook URL **production** của một instance n8n đang chạy thật, qua biến môi trường
   `B6_WEBHOOK_ANGLES/GENERATE/APPROVE` — đúng 3 ô mà `app-duyet-solution.html` cũng cần.
2. **Không so khớp chuỗi 1-1.** Nội dung sinh bằng LLM thật không tất định — kỳ vọng chỉ kiểm bất biến cấu trúc
   (số lượng, có/không field, brief_id echo đúng, không tự bịa chân dung/ý tưởng ngoài input).

**Output baked-in dưới mỗi cell là output THẬT**, chạy một lần trên instance thử nghiệm của người viết tài
liệu ngày 2026-08-10 — không phải số liệu bịa. Nhưng đó là instance của người viết, không phải của bạn.

**Trước khi Run All — bắt buộc:** mở Step 0, set 3 biến `B6_WEBHOOK_ANGLES/GENERATE/APPROVE` trỏ về **workflow
của chính bạn** (URL production lấy từ TH4a). Notebook **không có giá trị mặc định nào** — thiếu biến sẽ báo
lỗi rõ ràng ngay ở Step 0, chứ không âm thầm gọi nhầm vào instance của người khác.

### ⚙️ Step 0: Kết nối tới n8n thật (không tự-launch được)
Khai 3 webhook URL production — lấy từ n8n sau khi dựng xong TH4a (`checkpoint-bt4.md` Phần A), đúng giá trị
dán vào `app-duyet-solution.html`.

In [ ]:
import os, sys, json
from pathlib import Path

TEST_DIR = Path('.').resolve()
if str(TEST_DIR) not in sys.path:
    sys.path.insert(0, str(TEST_DIR))

# BẮT BUỘC điền 3 dòng dưới trước khi Run All -- KHÔNG có giá trị mặc định.
# Lấy đúng 3 URL production của WORKFLOW CỦA BẠN (dựng ở TH4a, checkpoint-bt4.md Phần A) --
# đúng giá trị bạn sẽ dán vào app-duyet-solution.html. Thiếu dòng nào, ô này sẽ báo lỗi rõ ràng
# ngay tại đây thay vì âm thầm gọi nhầm vào instance của người khác.
# os.environ['B6_WEBHOOK_ANGLES']   = 'https://<n8n-cua-ban>/webhook/b6/angles'
# os.environ['B6_WEBHOOK_GENERATE'] = 'https://<n8n-cua-ban>/webhook/b6/generate'
# os.environ['B6_WEBHOOK_APPROVE']  = 'https://<n8n-cua-ban>/webhook/b6/approve'

from interactive_b6_runner import B6Runner, post

runner = B6Runner()  # raise RuntimeError rõ ràng nếu thiếu biến môi trường ở trên
print('=' * 75)
print('STEP 0: WEBHOOK N8N THAT DA SAN SANG')
print('=' * 75)
for k, v in runner.urls.items():
    print(f'  {k:10s} -> {v}')

### 🧩 Step 1: Xem cấu trúc pipeline (đọc offline từ checkpoint)
Không login REST API n8n (không cần, webhook không đòi auth) — đọc thẳng `n8n-content-engine-solution.json`
đã track trong git để liệt kê node thật, kể cả node IF chặn thiếu-angle mới thêm ngày 2026-08-10.

In [ ]:
wf = json.loads((TEST_DIR.parent / 'checkpoints' / 'n8n-content-engine-solution.json').read_text(encoding='utf-8'))
print('=' * 75)
print(f"STEP 1: {wf['name']} -- {len(wf['nodes'])} node")
print('=' * 75)
for n in wf['nodes']:
    if n['type'] == 'n8n-nodes-base.stickyNote':
        continue
    print(f"* {n['name']}  |  {n['type']}")

STEP 1: [B6-TEST] Content Engine Sunrise Kids -- 37 node
* Webhook: Sinh nội dung  |  n8n-nodes-base.webhook
* Lớp 1 — Sinh ý tưởng  |  @n8n/n8n-nodes-langchain.chainLlm
* Parser: Ý tưởng  |  @n8n/n8n-nodes-langchain.outputParserStructured
* Google Gemini Chat Model  |  @n8n/n8n-nodes-langchain.lmChatGoogleGemini
* Lớp 2 — Viết nội dung  |  @n8n/n8n-nodes-langchain.chainLlm
* Parser: Nội dung  |  @n8n/n8n-nodes-langchain.outputParserStructured
* Lớp 3 — Seeding & image brief  |  @n8n/n8n-nodes-langchain.chainLlm
* Parser: Seeding & ảnh  |  @n8n/n8n-nodes-langchain.outputParserStructured
* Lớp 3b — Gửi yêu cầu tạo ảnh (GeminiGen)  |  n8n-nodes-base.httpRequest
* Gộp kết quả  |  n8n-nodes-base.set
* Lớp 4 — Ghi Content_Queue  |  n8n-nodes-base.googleSheets
* Trả về app  |  n8n-nodes-base.respondToWebhook
* Webhook: Duyệt  |  n8n-nodes-base.webhook
* Chuẩn hóa quyết định  |  n8n-nodes-base.set
* Cập nhật Content_Queue  |  n8n-nodes-base.googleSheets
* Ghi Publish_Log  |  n8n-nodes-base.go

### 💡 Step 2 (Demo Type 1): Sinh 5 ý tưởng từ brief Sunrise Kids
Gọi `/b6/angles` — chỉ chạy Lớp 1, ~10 giây, chưa tốn phí ảnh. Kỳ vọng: 5 ý tưởng, phủ ≥2 chân dung,
`brief_id` echo đúng.

In [ ]:
angles_payload = {
    'brief_id': 'SUNRISE-KIDS-TUYENSINH-2026-08',
    'brief': ("Trung tam Anh ngu Sunrise Kids, 2 co so TP.HCM (Go Vap, Tan Binh). Day tieng Anh tre tieu hoc 6-11 tuoi.\n"
              "Khoa 'Tieng Anh giao tiep nen tang': 3 buoi/tuan, 75 phut/buoi, 4 cap do. Lop toi da 15 be, "
              "mot giao vien ban ngu + mot tro giang nguoi Viet theo ca buoi.\n"
              "KHONG cam ket diem so hay chung chi. KHONG day luyen thi. KHONG nhan be duoi 6 tuoi.\n"
              "Kenh dang: Fanpage va TikTok. Muc tieu: phu huynh nhan tin hoi va dang ky hoc thu."),
    'personas': ("PH1 - Me be lop 1-2. Sot ruot vi con hoc o truong ma ve nha khong noi duoc.\n"
                 "PH2 - Phu huynh con lop 4-5. Sap het cap mot, tieng Anh la phan con yeu nhat.\n"
                 "PH3 - Phu huynh da that vong o trung tam khac vi lop dong, con khong tien bo.")
}
angles_result = post(runner.urls['angles'], angles_payload)
print(json.dumps(angles_result, ensure_ascii=False, indent=2))

{
  "brief_id": "SUNRISE-KIDS-TUYENSINH-2026-08",
  "brief_title": "Tuyển sinh khóa Tiếng Anh giao tiếp nền tảng cho tiểu học",
  "personas_covered": [
    "PH1",
    "PH2",
    "PH3"
  ],
  "angles": [
    {
      "angle_id": "ANG-001",
      "nhom": "Giao tiếp thực tế",
      "y_tuong": "Video ngắn trên TikTok quay cảnh lớp học thực tế: Giáo viên bản ngữ cùng bé tương tác, trợ giảng người Việt hỗ trợ sát sao. Caption: Giúp con tự tin nói tiếng Anh sau giờ học ở trường. Mời ba mẹ đăng ký học thử tại Gò Vấp/Tân Bình.",
      "chan_dung": "PH1",
      "muc_tieu": "CONVERSION",
      "kenh_phu_hop": [
        "TikTok"
      ]
    },
    {
      "angle_id": "ANG-002",
      "nhom": "Tư duy ngôn ngữ",
      "y_tuong": "Bài viết Fanpage: Đừng chỉ học để thi, hãy học để dùng. Chia sẻ về phương pháp 3 buổi/tuần, mỗi buổi 75 phút tại Sunrise Kids giúp con hình thành phản xạ tự nhiên thay vì áp lực điểm số.",
      "chan_dung": "PH2",
      "muc_tieu": "EDUCATION",
      "kenh_phu_hop": [
     

### 🐛 Step 3 (Demo Type 2 — kể chuyện bug thật): Gọi `/b6/generate` mà THIẾU `angle`
Đây là ca quan trọng nhất notebook: **live test từng bắt được 1 bug thật ở đây** (2026-08-10) — thiếu `angle`
không hề báo lỗi, LLM tự **bịa** `angle_id` (vd `"TUYEN-SINH-MAM-NON"`) rồi viết cả bài từ ý tưởng không ai
chọn, vi phạm đúng nguyên tắc lõi cả buổi "không được tự bịa". Test tĩnh (đọc prompt, suy đoán n8n sẽ tự lỗi)
đã SAI. Đã sửa bằng node IF thật (`Có angle hợp lệ?`) chặn trước Lớp 2 — giờ trả **HTTP 400 rõ ràng**.

In [ ]:
import urllib.request, urllib.error
req = urllib.request.Request(
    runner.urls['generate'],
    data=json.dumps({'brief_id': 'X', 'brief': 'x', 'brand_voice': 'x'}).encode('utf-8'),
    headers={'Content-Type': 'application/json'}, method='POST')
try:
    with urllib.request.urlopen(req, timeout=30) as resp:
        print('HTTP', resp.status, resp.read().decode())
except urllib.error.HTTPError as e:
    print('HTTP', e.code, '(dung ky vong -- loi ro rang, khong phai 200 rong hay bia angle_id)')
    print(e.read().decode())

HTTP 400 (dung ky vong -- loi ro rang, khong phai 200 rong hay bia angle_id)
{"error":"Thieu 'angle' (y tuong da chon) trong body cua /b6/generate. Goi /b6/angles truoc, chon 1 y tuong roi gui nguyen object angle len — khong duoc tu chon thay."}


### 💸 Step 4 (Demo Type 3 — tuỳ chọn, TỐN PHÍ): Đường đầy đủ Lớp 2→3→4
Chạy hết chuỗi viết bài + seeding + sinh ảnh thật (~2 phút, tốn phí GeminiGen.ai) + cả 2 Judge. **Mặc định
TẮT** (`CHAY_FULL = False`) để không tốn phí khi Run All ngoài giờ dạy — GV bật `True` khi thật sự muốn demo
trên lớp.

In [ ]:
CHAY_FULL = False  # doi thanh True khi muon demo that tren lop -- ton phi anh + ~2 phut

if not CHAY_FULL:
    print('SKIP -- dat CHAY_FULL = True o o tren roi chay lai o nay de demo duong day du (ton phi anh that).')
else:
    angle_da_chon = angles_result['angles'][0]
    payload = {
        'brief_id': 'SUNRISE-KIDS-TUYENSINH-2026-08',
        'brief': angles_payload['brief'],
        'brand_voice': ("Noi chuyen nhu nguoi cung dang co con di hoc. Binh tinh, cu the, khong len gan.\n"
                         "CAM: cam ket ket qua; noi dung luyen thi/chung chi/diem so; hu doa phu huynh; "
                         "neu ten doi thu; so lieu khong co trong brief; anh tre em that; cau tuyet doi; qua 2 emoji."),
        'angle': angle_da_chon
    }
    out = post(runner.urls['generate'], payload, timeout=180)
    print(f"source_angle_id: {out.get('assets', {}).get('source_angle_id')} (da chon: {angle_da_chon['angle_id']})")
    print(f"image_ok: {out.get('image_ok')}")
    print(f"judge_van_phong: {out.get('judge_van_phong')}")
    print(f"judge_anh: {out.get('judge_anh')}")

SKIP -- dat CHAY_FULL = True o o tren roi chay lai o nay de demo duong day du (ton phi anh that).


### ✅ Step 5 (Demo): Duyệt hợp lệ + gửi status cấm bị chặn
Ca 1: duyệt bình thường → nhận `log_id` thật. Ca 2: cố tình gửi `status: "Published"` (giá trị cấm) → n8n phải
tự **clamp về `"Needs Review"`**, không bao giờ echo lại `Published` (buổi học cố tình không có bước đăng bài).

In [ ]:
approve_ok = post(runner.urls['approve'], {
    'post_id': 'NOTEBOOK-DEMO-01', 'status': 'Approved',
    'noi_dung': 'Bai demo notebook -- khong phai noi dung that.',
    'nguoi_duyet': 'GV Demo', 'ghi_chu': 'Chay tu 06_content_engine_lab_demo.ipynb'
})
print('Duyet hop le:', json.dumps(approve_ok, ensure_ascii=False))

approve_clamp = post(runner.urls['approve'], {
    'post_id': 'NOTEBOOK-DEMO-02', 'status': 'Published',
    'noi_dung': 'Bai demo notebook -- kiem status cam bi chan.',
    'nguoi_duyet': 'GV Demo', 'ghi_chu': 'Chay tu 06_content_engine_lab_demo.ipynb'
})
print('Gui status cam:', json.dumps(approve_clamp, ensure_ascii=False))
assert approve_clamp['status'] != 'Published', 'BUG: Published lot qua!'
print('Xac nhan: status Published KHONG lot qua, da bi clamp ve', approve_clamp['status'])

Duyet hop le: {"ok": true, "post_id": "NOTEBOOK-DEMO-01", "status": "Approved", "log_id": "LOG-260810091856"}
Gui status cam: {"ok": true, "post_id": "NOTEBOOK-DEMO-02", "status": "Needs Review", "log_id": "LOG-260810091901"}
Xac nhan: status Published KHONG lot qua, da bi clamp ve Needs Review


### 📋 Step 6: Chạy toàn bộ `test-cases.json` (8 case, bỏ qua case tốn phí)
Gọi lại `interactive_b6_runner.py` như dòng lệnh thật GV/HV sẽ chạy trước mỗi buổi dạy — không cần `--full`
cho demo nhanh.

In [ ]:
import subprocess
result = subprocess.run(
    [sys.executable, 'interactive_b6_runner.py'],
    cwd=str(TEST_DIR), capture_output=True, text=True, encoding='utf-8', env=os.environ)
print(result.stdout)
if result.returncode != 0:
    print('--- STDERR ---')
    print(result.stderr)

Gọi webhook thật tại: angles=https://n8n-qns0.srv1741374.hstgr.cloud/webhook/b6/angles
                       generate=https://n8n-qns0.srv1741374.hstgr.cloud/webhook/b6/generate
                       approve=https://n8n-qns0.srv1741374.hstgr.cloud/webhook/b6/approve

▶ TC01 — Brief đầy đủ Sunrise Kids (đúng NGUYEN_LIEU trong app-duyet-solution.html) — ca chuẩn
    ✓ số ý tưởng đúng kỳ vọng — nhận 5, kỳ vọng 5
    ✓ brief_id echo đúng đầu vào — nhận 'SUNRISE-KIDS-TUYENSINH-2026-08'
    ✓ đủ số chân dung tối thiểu được phủ — phủ ['PH1', 'PH2', 'PH3']

▶ TC02 — Personas đầu vào CHỈ có PH1 — kiểm workflow không tự bịa PH2/PH3 không có trong input
    ✓ số ý tưởng đúng kỳ vọng — nhận 5, kỳ vọng 5
    ✓ brief_id echo đúng đầu vào — nhận 'SUNRISE-KIDS-1PERSONA-TEST'
    ✓ không tự bịa chân dung ngoài input

▶ TC03 — Thương hiệu KHÁC hẳn (không phải trung tâm Anh ngữ) — kiểm workflow không hardcode riêng Sunrise Kids
    ✓ số ý tưởng đúng kỳ vọng — nhận 5, kỳ vọng 5
    ✓ brief_id echo đúng 

### 🌐 Step 7: Mở app duyệt (`app-duyet-solution.html`) qua web server + IFrame
Giống Step 10 của notebook B5 — bật server cục bộ để xem app thật ngay trong notebook. App cần bạn tự dán 3
URL webhook (không có URL nào hardcode trong app, đúng thiết kế TH4b).

In [ ]:
import socket, threading, http.server, socketserver
from IPython.display import IFrame, display, HTML

PORT = 8086
SERVE_DIR = TEST_DIR.parent  # de URL .../checkpoints/app-duyet-solution.html hoat dong

def is_port_open(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(0.5)
        return s.connect_ex(('127.0.0.1', port)) == 0

if not is_port_open(PORT):
    class ThreadedHTTPServer(socketserver.ThreadingMixIn, http.server.HTTPServer):
        daemon_threads = True
        allow_reuse_address = True

    class Handler(http.server.SimpleHTTPRequestHandler):
        def __init__(self, *a, **kw):
            super().__init__(*a, directory=str(SERVE_DIR), **kw)

    server = ThreadedHTTPServer(('0.0.0.0', PORT), Handler)
    threading.Thread(target=server.serve_forever, daemon=True).start()

app_url = f'http://localhost:{PORT}/checkpoints/app-duyet-solution.html'
print('App duyet:', app_url)
print('Dan 3 URL webhook o Step 0 vao phan Cau hinh trong app roi bam Sinh y tuong.')
display(IFrame(src=app_url, width='100%', height=760))